In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.case_registry import cases
from pull_temple_data import fetch_all_signals

In [ ]:
for case in cases:
    temple_path = os.path.join('data/temple/', os.path.basename(case['polar_path']))
    fetch_all_signals(
        user_id=case['user_id'],
        start_time=case['start_time'],
        end_time=case['end_time'],
        save_path=temple_path,
    )

In [ ]:
def fetch_polar_data(polar_date, file_path) -> pd.DataFrame:
    # Check the first row to determine format
    first_row = pd.read_csv(file_path, nrows=1)
    
    # New format: has 'timestamp' column directly
    if 'timestamp' in first_row.columns:
        # New format: timestamp,hr_bpm,rr_intervals_ms,contact_status
        df = pd.read_csv(file_path)
        polar_df = df[['timestamp', 'hr_bpm']].copy()
        polar_df.rename(columns={'hr_bpm': 'hr'}, inplace=True)
        
        # Parse timestamp (format: "2026-01-29 09:25:08 +0000")
        # Timestamp is in UTC, convert to Asia/Kolkata to match old format behavior
        polar_df["actual_date_time"] = pd.to_datetime(polar_df['timestamp']).dt.tz_convert('Asia/Kolkata')
        polar_df["actual_time"] = polar_df["actual_date_time"].dt.time
        polar_df["timestamp"] = polar_df["actual_date_time"].astype('int64') // 10**6
        
        # Add cadence column if not present (set to NaN for new format)
        if 'cadence' not in polar_df.columns:
            polar_df['cadence'] = None
        
    else:
        # Old format: has metadata rows with 'Start time' and 'Duration'
        metadata = first_row
        polar_start_time = metadata['Start time'].iloc[0]
        duration = metadata['Duration'].iloc[0]
        polar_sampling_rate = 1 # manually set to 1 Hz
        
        df = pd.read_csv(file_path, header=2)
        # Handle both possible column names
        if 'hr_bpm' in df.columns:
            selected_data = df[['Time', 'hr_bpm', 'Cadence']].copy()
            selected_data.rename(columns={'Time': 'time', 'hr_bpm': 'hr', 'Cadence': 'cadence'}, inplace=True)
        else:
            selected_data = df[['Time', 'HR (bpm)', 'Cadence']].copy()
            selected_data.rename(columns={'Time': 'time', 'HR (bpm)': 'hr', 'Cadence': 'cadence'}, inplace=True)
        
        polar_df = selected_data.copy()
        start_dt = pd.to_datetime(f"{polar_date} {polar_start_time}")
        new_time_axis = pd.date_range(
            start=start_dt,
            periods=len(polar_df),
            freq=f"{polar_sampling_rate}S"
        )
        polar_df["actual_date_time"] = new_time_axis.tz_localize('Asia/Kolkata')
        polar_df["actual_time"] = polar_df["actual_date_time"].dt.time
        polar_df["timestamp"] = polar_df["actual_date_time"].astype('int64') // 10**6
    
    # Filter out invalid HR values
    polar_df = polar_df[polar_df["hr"] != 0]
    polar_df = polar_df[polar_df["hr"].notna()]
    return polar_df

In [ ]:
for case in cases:
    polar_path = os.path.join('data/polar/', os.path.basename(case['polar_path']))
    temple_path = os.path.join('data/temple/', os.path.basename(case['polar_path']))
    if not os.path.exists(polar_path) or not os.path.exists(temple_path):
        continue
    
    temple_df = pd.read_csv(temple_path)
    g = temple_df.loc[temple_df['green'].notna(), ['timestamp', 'green']]
    a = temple_df.loc[temple_df['accel_z'].notna(), ['timestamp', 'accel_x', 'accel_y', 'accel_z']]
    ga = pd.merge_asof(
        g.sort_values('timestamp'),
        a.sort_values('timestamp'),
        on='timestamp',
        tolerance=60,
        direction='nearest',
    ).dropna(subset=['accel_z']).reset_index(drop=True)

    polar_df = fetch_polar_data(case['polar_date'], polar_path)
    h = polar_df.loc[polar_df['hr'].notna(), ['timestamp', 'hr']]
    combined_df = pd.merge_asof(
        ga.sort_values('timestamp'),
        h.sort_values('timestamp'),
        on='timestamp',
        tolerance=1000,
        direction='nearest',
    ).dropna(subset=['hr']).reset_index(drop=True)

    combined_path = os.path.join('data/combined/', os.path.basename(case['polar_path']))
    if len(combined_df) > 0:
        combined_df.to_csv(combined_path, index=False)
        print(f"✓ Saved: {combined_path} ({len(ga)}&{len(h)}:{len(combined_df)} rows)")

---

In [2]:
import os
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pull_temple_data import (
    fetch_all_signals,
    fetch_sleep_windows_for_user,
    get_user_id_by_email,
)

In [ ]:
EMAILS_PATH = "sleep/source_emails.txt"
START_DATE = "2026-02-01"
END_DATE = "2026-05-25"
SLEEP_OUTPUT_DIR = "sleep/temple"
MAX_WORKERS = 6  # parallel users; ClickHouse instance can handle this

emails = [
    line.strip() for line in Path(EMAILS_PATH).read_text().splitlines()
    if line.strip() and not line.strip().startswith("#")
]

In [ ]:
tasks = []
for email in emails:
    user_id = get_user_id_by_email(email)
    if not user_id:
        print(f"✗ no user_id for {email}")
        continue
    safe_email = email.replace("@", "_at_").replace("+", "_plus_").replace(".", "_")
    save_path = os.path.join(SLEEP_OUTPUT_DIR, f"{safe_email}.csv")
    tasks.append((email, user_id, save_path))

print(f"\nFetching {len(tasks)} users with {MAX_WORKERS} workers …\n")

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {
        ex.submit(
            fetch_sleep_windows_for_user,
            user_id, save_path, START_DATE, END_DATE,
        ): email
        for email, user_id, save_path in tasks
    }
    for fut in as_completed(futures):
        email = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"✗ {email}: {e}")

In [ ]:
from datetime import timedelta

REQUIRED_COLS = ['green', 'accel_x', 'accel_y', 'accel_z']

for filepath in glob.glob('sleep/temple/*.csv'):
    user_stem = os.path.splitext(os.path.basename(filepath))[0]

    temple_df = pd.read_csv(filepath)
    if temple_df.empty:
        continue

    g = temple_df.loc[temple_df['green'].notna(), ['timestamp', 'green']]
    a = temple_df.loc[temple_df['accel_z'].notna(), ['timestamp', 'accel_x', 'accel_y', 'accel_z']]
    ga = pd.merge_asof(
        g.sort_values('timestamp'),
        a.sort_values('timestamp'),
        on='timestamp',
        tolerance=60,
        direction='nearest',
    ).dropna(subset=['accel_z']).reset_index(drop=True)
    if ga.empty:
        continue

    ga.to_csv(filepath, index=False)

In [5]:
with open('sleep/select.txt', 'w') as f:
    for filepath in glob.glob('sleep/data/*.csv'):
        data = pd.read_csv(filepath)
        if data.empty or not {'accel_x', 'accel_y', 'accel_z'}.issubset(data.columns):
            continue
        data = data.sort_values('timestamp').reset_index(drop=True)
        data['timestamp_ist'] = pd.to_datetime(data['timestamp'], unit='ms', utc=True).dt.tz_convert('Asia/Kolkata')

        accel_mag = np.sqrt(data['accel_x']**2 + data['accel_y']**2 + data['accel_z']**2)
        accel_mag = pd.Series(accel_mag.to_numpy(), index=pd.DatetimeIndex(data['timestamp_ist']))
        roll = accel_mag.rolling('15s')
        std = roll.std().to_numpy()
        count = roll.count().to_numpy()

        quiet_mask = (count > 100) & (std < 250.0)
        for ts in data.loc[quiet_mask, 'timestamp_ist']:
            f.write(f'{filepath}: {ts}\n')

In [7]:
seen = set()
with open('sleep/select.txt', 'r') as f:
    lines = f.readlines()

with open('sleep/select.txt', 'w') as f:
    for line in lines:
        path, ts = line.rstrip('\n').split(': ', 1)
        key = f'{path}: {ts[:19]}'
        if key not in seen:
            seen.add(key)
            f.write(key + '\n')